In [75]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [76]:
# Import libraries
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
pd.set_option("display.max_columns", None)
import config

from sklearn.feature_selection import (
    VarianceThreshold,
    mutual_info_classif
)

from feature_engine.selection import (
    DropCorrelatedFeatures
)

In [77]:
# Reading data from source
train = pd.read_csv(config.TRAIN_DATA_PATH)

In [78]:
# The Photometry (Light Filters)
photometry_cols = ["u", "g", "r", "i", "z"]

# The Astrophysics (Distance)
astrophysics_cols = ["redshift"]

# The Sky Coordinates (Position)
coordinate_cols = ["alpha", "delta"]

# The Sub-Categories
categorical_cols = ["spectral_type", "galaxy_population"]

# Target feature
target_col = 'class'

In [79]:
train.columns

Index(['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
       'spectral_type', 'galaxy_population', 'class'],
      dtype='object')

### Feature engineering

In [80]:
def fe(df):
    # Calculate consecutive color differences (adjacent bands)
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]

    # Calculate crucial extended color differences
    df["u_r"] = df["u"] - df["r"]
    df["u_i"] = df["u"] - df["i"]

    df["redshift_g_r"] = df["redshift"] * df["g_r"]
    df["redshift_u_g"] = df["redshift"] * df["u_g"]

    alpha_rad = np.radians(df["alpha"])
    delta_rad = np.radians(df["delta"])

    df["coord_X"] = np.cos(delta_rad) * np.cos(alpha_rad)
    df["coord_Y"] = np.cos(delta_rad) * np.sin(alpha_rad)
    df["coord_Z"] = np.sin(delta_rad)

    df["coord_radial_dist"] = np.sqrt(df["alpha"] ** 2 + df["delta"] ** 2)

    # deviation from mean redshift
    df["redshift_dev_from_galaxy_population"] = df["redshift"] - df.groupby("galaxy_population")["redshift"].transform("mean")
    df["redshift_dev_from_spectral_type"] = df["redshift"] - df.groupby("spectral_type")["redshift"].transform("mean")

    # Color per distance
    for c in ["u_g", "g_r", "r_i", "i_z"]:
        df[f"{c}_per_redshift"] = (df[c] / (df["redshift"].abs() + 1e-6)).astype("float32")

    # Descriptive stats
    bands = ["u", "g", "r", "i", "z"]
    df["mag_std"] = df[bands].std(axis=1).astype("float32")
    df["mag_var"] = df[bands].var(axis=1).astype("float32")
    df["mag_skew"] = df[bands].skew(axis=1).astype("float32")
    df["mag_kurt"] = df[bands].kurt(axis=1).astype("float32")
    df["mag_ptp"] = (df[bands].max(axis=1) - df[bands].min(axis=1)).astype("float32")

    df["g_rest_proxy"] = (df["g"] / (1 + df["redshift"])).astype("float32")
    df["r_rest_proxy"] = (df["r"] / (1 + df["redshift"])).astype("float32")

    # category interaction
    df['galaxy_population_x_spectral_type'] = df['galaxy_population'] + "x" + df["spectral_type"]
    df['galaxy_population_x_spectral_type'] = df['galaxy_population_x_spectral_type'].astype('category')
    # Factorize
    df['galaxy_population_x_spectral_type'], _ = pd.factorize(df['galaxy_population_x_spectral_type'])

    df['alpha+delta'] = (df['alpha'] + df['delta']).astype('float32')
    df['alpha-delta'] = (df['alpha'] - df['delta']).astype('float32')
    df['alpha%1'] = (df['alpha'] % 1).astype('float32')
    df['delta%1'] = (df['delta'] % 1).astype('float32')

    df["blue_curvature"] = (df["u"] - 2 * df["g"] + df["r"]).astype("float32")
    df["red_curvature"] = (df["r"] - 2 * df["i"] + df["z"]).astype("float32")

    agg_targets = ["redshift", "u_g"]
    categories = ["galaxy_population", "spectral_type"]

    for cat in categories:
        for target in agg_targets:
            # Group stats maps
            grouped = df.groupby(cat)[target]

            # Median Deviation (Robust to cosmic outliers)
            df[f"{target}_dev_median_{cat}"] = (
                df[target] - grouped.transform("median")
            ).astype("float32")

            # Range Extremes
            df[f"{target}_min_diff_{cat}"] = (
                df[target] - grouped.transform("min")
            ).astype("float32")
            df[f"{target}_max_diff_{cat}"] = (
                grouped.transform("max") - df[target]
            ).astype("float32")

            # Standard Deviation
            group_std = grouped.transform("std") + 1e-6
            df[f"{target}_zscore_{cat}"] = (
                (df[target] - grouped.transform("mean")) / group_std
            ).astype("float32")

    # Group Count / Frequency Encoding
    for col in categories + ["galaxy_population_x_spectral_type"]:
        df[f"{col}_counts"] = df.groupby(col)[col].transform("count").astype("int32")

    combo_col = "galaxy_population_x_spectral_type"

    for target in ["redshift", "mag_ptp", "blue_curvature"]:
        grouped_combo = df.groupby(combo_col)[target]

        df[f"{target}_mean_by_combo"] = grouped_combo.transform("mean").astype(
            "float32"
        )
        df[f"{target}_dev_median_by_combo"] = (
            df[target] - grouped_combo.transform("median")
        ).astype("float32")

    df["alpha_bin"] = df["alpha"].round(0)
    df["delta_bin"] = df["delta"].round(0)

    spatial_group = df.groupby(["alpha_bin", "delta_bin"])["redshift"]
    df["spatial_cluster_redshift_mean"] = spatial_group.transform("mean").astype(
        "float32"
    )
    df["spatial_cluster_redshift_std"] = spatial_group.transform("std").fillna(0).astype(
        "float32"
    )

    df = df.drop(["alpha_bin", "delta_bin"], axis=1)

    combo_counts = df.groupby(combo_col)[combo_col].transform("count")
    galaxy_pop_counts = (
        df.groupby("galaxy_population")["galaxy_population"].transform("count")
        + 1e-6
    )

    df["spectral_type_share_in_galaxy_pop"] = (
        combo_counts / galaxy_pop_counts
    ).astype("float32")

    for cat in ["galaxy_population", "spectral_type"]:
        df[f"r_magnitude_dev_mean_{cat}"] = (
            df["r"] - df.groupby(cat)["r"].transform("mean")
        ).astype("float32")

    # Drop irrelevant cols
    df = df.drop(['galaxy_population', 'spectral_type'], axis=1)

    return df

train = fe(train)

X = train.drop(['id', 'class'], axis=1)
y = train['class']

### Feature Selection

1. Variance Threshold

In [81]:
selector = VarianceThreshold(0.01).set_output(transform='pandas')
train_selected = selector.fit_transform(X)

dropped_mask = ~selector.get_support()
dropped_cols = X.columns[dropped_mask].tolist()

print(f"Dropped {len(dropped_cols)} columns:")
print(dropped_cols)

Dropped 0 columns:
[]


2. Correlated Features

In [82]:
selector = DropCorrelatedFeatures(
    method='pearson',
    threshold=0.99,
    missing_values='raise'
)

train_selected = selector.fit_transform(X)
print(selector.correlated_feature_dict_)

{'alpha': {'coord_radial_dist'}, 'coord_Z': {'delta'}, 'g_rest_proxy': {'r_rest_proxy'}, 'galaxy_population_counts': {'u_g_min_diff_galaxy_population'}, 'galaxy_population_x_spectral_type_counts': {'spectral_type_share_in_galaxy_pop'}, 'mag_ptp': {'mag_std'}, 'redshift': {'redshift_max_diff_galaxy_population', 'redshift_min_diff_galaxy_population', 'redshift_min_diff_spectral_type', 'redshift_max_diff_spectral_type'}, 'redshift_dev_from_galaxy_population': {'redshift_dev_median_galaxy_population'}}


In [83]:
selector = DropCorrelatedFeatures(
    method='spearman',
    threshold=0.99,
    missing_values='raise'
)

train_selected = selector.fit_transform(X)
print(selector.correlated_feature_dict_)

{'alpha': {'coord_radial_dist'}, 'coord_Z': {'delta'}, 'g_rest_proxy': {'r_rest_proxy'}, 'mag_ptp': {'mag_var', 'mag_std'}, 'redshift': {'redshift_max_diff_galaxy_population', 'redshift_min_diff_galaxy_population', 'redshift_min_diff_spectral_type', 'redshift_max_diff_spectral_type'}, 'redshift_dev_from_galaxy_population': {'redshift_zscore_galaxy_population'}, 'u_g_dev_median_galaxy_population': {'u_g_zscore_galaxy_population'}, 'u_g_dev_median_spectral_type': {'u_g_zscore_spectral_type'}}


3. Mutual Information

In [58]:
mask = X.columns.isin([
    'galaxy_population_x_spectral_type', 
    'galaxy_population_counts', 
    'spectral_type_counts', 
    'galaxy_population_x_spectral_type_counts'
])

mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=mask,
    random_state=42,
    n_neighbors=3,
    n_jobs=-1
)

mi = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

In [59]:
mi

redshift                                    0.514989
redshift_min_diff_galaxy_population         0.514590
redshift_min_diff_spectral_type             0.514349
redshift_max_diff_spectral_type             0.512512
redshift_max_diff_galaxy_population         0.510129
redshift_dev_median_galaxy_population       0.504500
redshift_dev_from_galaxy_population         0.476750
r_rest_proxy                                0.466443
redshift_dev_from_spectral_type             0.452115
g_rest_proxy                                0.447247
redshift_zscore_galaxy_population           0.432688
redshift_dev_median_spectral_type           0.430541
g_r_per_redshift                            0.421865
r_i_per_redshift                            0.415325
mag_std                                     0.402875
mag_var                                     0.402858
u_g_per_redshift                            0.396881
mag_ptp                                     0.394403
u_g_min_diff_spectral_type                  0.

4. Boruta

In [57]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=4, n_estimators=400, random_state=42)
selector = BorutaPy(rf, n_estimators='auto', verbose=2, random_state=42, early_stopping=True, n_iter_no_change=10)
selector.fit(X, y)

Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	37
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	37
Rejected: 	0
Early stopping: 2 out of 20
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	37
Rejected: 	0
Early stopping: 3 out of 20
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	37
Rejected: 	0
Early stopping: 4 out of 20
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	37
Rejected: 	0
Early stopping: 5 out of 20
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	37
Rejected: 	0
Early stopping: 6 out of 20
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	37
Rejected: 	0
Early stopping: 7 out of 20
Iteration: 	8 / 100
Confirmed: 	35
Tentative: 	2
Rejected: 	0
Iteration: 	9 / 100
Confirmed: 	35
Tentative: 	2
Rejected: 	0
Early stopping: 2 out of 20
Iteration: 	10 / 100
Confirmed: 	35
Tentative: 	2
Rejected: 	0
Early stopping: 3 out of 20
Iteration: 	11 / 100
Confirmed: 	35
Tentative: 	2
Rejected: 	0
Early stopping: 4 out of 20
Iteration: 	12 / 100
Confirmed: 	35
Tentative: 	2
Rejected: 	0
E

BorutaPy(early_stopping=True,
         estimator=RandomForestClassifier(class_weight='balanced', max_depth=4,
                                          n_estimators=215, n_jobs=-1,
                                          random_state=RandomState(MT19937) at 0x1FC2E184440),
         n_estimators='auto',
         random_state=RandomState(MT19937) at 0x1FC2E184440, verbose=2)